# Translation

In [1]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [2]:
!pip install evaluate sacrebleu transformers datasets accelerate sacremoses

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 31.3 MB/s eta 0:00:00


In [3]:
import pandas as pd
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)
from tqdm import tqdm
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset
import evaluate
from sklearn.model_selection import train_test_split
import numpy as np
import torch.nn as nn
import torch.optim as optim
import random
from collections import Counter

In [14]:
# df_translation = pd.read_csv("https://drive.google.com/uc?id=1ufEzybfWMXJh4W_AFxQp_c2mmGESRDT2")
# train_df, temp_df = train_test_split(df_translation, test_size=0.2, random_state=42)
# val_df, test_df = train_test_split(temp_df, test_size=(200/1200), random_state=42)
# train_df.to_csv('train.csv', index=False)
# val_df.to_csv('val.csv', index=False)
# test_df.to_csv('test.csv', index=False)

train_df = pd.read_csv('train2.csv')
val_df = pd.read_csv('val2.csv')
test_df = pd.read_csv('test_merged2.csv')

print(f"   Train Rows : {len(train_df)}")
print(f"   Val Rows   : {len(val_df)}")
print(f"   Test Rows  : {len(test_df)}")

   Train Rows : 4800
   Val Rows   : 1000
   Test Rows  : 200


In [5]:
# Calculate character length for each row
train_df['char_count'] = train_df['text'].astype(str).apply(len)

# Get Min and Max
min_chars = train_df['char_count'].min()
max_chars = train_df['char_count'].max()

print(f"Minimum Character Count: {min_chars}")
print(f"Maximum Character Count: {max_chars}")

# Show rows with Min and Max lengths
print("\n--- Shortest Text ---")
print(train_df[train_df['char_count'] == min_chars][['text', 'char_count']].head(1))

print("\n--- Longest Text ---")
print(train_df[train_df['char_count'] == max_chars][['text', 'char_count']].head(1))

# See full statistics (Mean, Std, Percentiles)
print("\n--- Statistics ---")
print(train_df['char_count'].describe())

Minimum Character Count: 4
Maximum Character Count: 367

--- Shortest Text ---
      text  char_count
3093  name           4

--- Longest Text ---
                                                   text  char_count
2167  tiket masuk 50ribu orang tapi worth itu sih ka...         367

--- Statistics ---
count    4800.000000
mean      153.078958
std        72.861249
min         4.000000
25%        92.000000
50%       136.000000
75%       203.000000
max       367.000000
Name: char_count, dtype: float64


In [4]:
class TranslationDataset(Dataset):
    def __init__(self, texts):
        self.texts = [str(t) if pd.notna(t) else "" for t in texts]

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return self.texts[idx]

In [ ]:
# --- OPTION B: NLLB (Meta - Higher Accuracy, Slower) ---
# MODEL_NAME = "facebook/nllb-200-distilled-600M"

# --- OPTION C: M2M100 (Meta - General Purpose) ---
# MODEL_NAME = "facebook/m2m100_418M"

In [5]:
def evaluate_translation(df, target_col, output_col, model_name):
    """
    Calculates and prints the SacreBLEU score for translation predictions.

    Args:
        df (pd.DataFrame): DataFrame containing predictions and ground truth.
        target_col (str): Name of the column with ground truth translations.
        output_col (str): Name of the column with model predictions.
        model_name (str): Name of the model for display purposes.
    """
    if target_col in df.columns:
        print("\nCalculating Evaluation Metrics...")
        bleu_metric = evaluate.load("sacrebleu")
        meteor_metric = evaluate.load("meteor")

        # Filter valid rows
        eval_df = df.dropna(subset=[target_col, output_col])

        preds = eval_df[output_col].tolist()
        refs = [[ref] for ref in eval_df[target_col].tolist()]

        if len(refs) > 0:
            bleu_results = bleu_metric.compute(predictions=preds, references=refs)
            meteor_results = meteor_metric.compute(predictions=preds, references=refs)
            print("-" * 30)
            print(f"Model: {model_name}")
            print(f"BLEU Score: {bleu_results['score']:.2f}")
            print(f"Meteor Score: {meteor_results['meteor']:.2f}")
            print("-" * 30)
        else:
            print("Ground truth column is empty.")
            return None
    else:
        print(f"Column '{target_col}' not found. Skipping evaluation.")
        return None

## LSTM

In [6]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [7]:
!wget https://dl.fbaipublicfiles.com/fasttext/vectors-wiki/wiki.id.vec

!wget https://dl.fbaipublicfiles.com/fasttext/vectors-wiki/wiki.en.vec

--2025-11-23 10:35:04--  https://dl.fbaipublicfiles.com/fasttext/vectors-wiki/wiki.id.vec
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 3.163.189.96, 3.163.189.14, 3.163.189.51, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|3.163.189.96|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 791484502 (755M) [binary/octet-stream]
Saving to: ‘wiki.id.vec’

wiki.id.vec         100%[===================>] 754.82M  63.9MB/s    in 12s     

2025-11-23 10:35:16 (62.6 MB/s) - ‘wiki.id.vec’ saved [791484502/791484502]

--2025-11-23 10:35:16--  https://dl.fbaipublicfiles.com/fasttext/vectors-wiki/wiki.en.vec
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 3.163.189.96, 3.163.189.14, 3.163.189.51, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|3.163.189.96|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6597238061 (6.1G) [binary/octet-stream]
Saving to: ‘wiki.en.vec’

wiki.en.vec        

In [8]:
class Vocabulary:
    def __init__(self):
        self.stoi = {"<PAD>": 0, "<SOS>": 1, "<EOS>": 2, "<UNK>": 3}
        self.itos = {0: "<PAD>", 1: "<SOS>", 2: "<EOS>", 3: "<UNK>"}
        self.freq_threshold = 1

    def __len__(self):
        return len(self.stoi)

    @staticmethod
    def tokenizer(text):
        return text.lower().split()

    def build_vocabulary(self, sentence_list):
        frequencies = Counter()
        idx = 4

        for sentence in sentence_list:
            for word in self.tokenizer(sentence):
                frequencies[word] += 1

                if frequencies[word] == self.freq_threshold:
                    self.stoi[word] = idx
                    self.itos[idx] = word
                    idx += 1

    def numericalize(self, text):
        tokenized_text = self.tokenizer(text)
        return [
            self.stoi[token] if token in self.stoi else self.stoi["<UNK>"]
            for token in tokenized_text
        ]

In [9]:
class ParallelDataset(Dataset):
    def __init__(self, df, src_col, trg_col, src_vocab, trg_vocab):
        self.df = df
        self.src_col = src_col
        self.trg_col = trg_col
        self.src_vocab = src_vocab
        self.trg_vocab = trg_vocab

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        src_text = self.df.iloc[index][self.src_col]
        trg_text = self.df.iloc[index][self.trg_col]

        # Convert text to indices
        src_indices = [self.src_vocab.stoi["<SOS>"]] + \
                      self.src_vocab.numericalize(src_text) + \
                      [self.src_vocab.stoi["<EOS>"]]

        trg_indices = [self.trg_vocab.stoi["<SOS>"]] + \
                      self.trg_vocab.numericalize(trg_text) + \
                      [self.trg_vocab.stoi["<EOS>"]]

        return torch.tensor(src_indices), torch.tensor(trg_indices)

# Collate function to pad batches to the same length
class MyCollate:
    def __init__(self, pad_idx):
        self.pad_idx = pad_idx

    def __call__(self, batch):
        src = [item[0] for item in batch]
        trg = [item[1] for item in batch]

        # Pad sequences
        src = torch.nn.utils.rnn.pad_sequence(src, batch_first=True, padding_value=self.pad_idx)
        trg = torch.nn.utils.rnn.pad_sequence(trg, batch_first=True, padding_value=self.pad_idx)

        return src, trg

In [10]:
def load_pretrained_embeddings(vocab, file_path, emb_dim=300):
    """
    Args:
        vocab: Your Vocabulary object (src_vocab or trg_vocab)
        file_path: Path to the .vec file (e.g., 'wiki.id.vec')
        emb_dim: Dimension of the vectors (FastText is usually 300)
    Returns:
        FloatTensor of shape (vocab_size, emb_dim)
    """
    print(f"Loading embeddings from {file_path}...")

    vocab_size = len(vocab)
    embedding_matrix = torch.randn(vocab_size, emb_dim)

    hit_count = 0
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        next(f)

        for line in tqdm(f, desc="Parsing Vectors"):
            parts = line.split()
            word = parts[0]

            if word in vocab.stoi:
                idx = vocab.stoi[word]
                try:
                    vector = torch.tensor([float(x) for x in parts[1:]])
                    if vector.shape[0] == emb_dim:
                        embedding_matrix[idx] = vector
                        hit_count += 1
                except:
                    continue

    print(f"Found {hit_count} / {vocab_size} words in pretrained file.")
    print(f"Coverage: {hit_count / vocab_size * 100:.2f}%")

    return embedding_matrix

In [11]:
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim, n_layers, dropout, pretrained_weights=None):
        super().__init__()
        self.hid_dim = hid_dim
        self.n_layers = n_layers

        self.embedding = nn.Embedding(input_dim, emb_dim)

        if pretrained_weights is not None:
            self.embedding.weight.data.copy_(pretrained_weights)
            self.embedding.weight.requires_grad = True

        # 1. Bidirectional
        self.lstm = nn.LSTM(emb_dim, hid_dim, n_layers,
                            dropout=dropout, batch_first=True,
                            bidirectional=True)

        self.dropout = nn.Dropout(dropout)

        self.fc_hidden = nn.Linear(hid_dim * 2, hid_dim)
        self.fc_cell = nn.Linear(hid_dim * 2, hid_dim)

    def forward(self, src):
        # src = [batch size, src len]
        embedded = self.dropout(self.embedding(src))

        # outputs = [batch size, src len, hid_dim * 2]
        # hidden = [n_layers * 2, batch size, hid_dim]
        # cell = [n_layers * 2, batch size, hid_dim]
        outputs, (hidden, cell) = self.lstm(embedded)

        # 3. Handle the Hidden/Cell States
        # We need to concatenate the forward and backward states of the LAST layer
        # hidden[-2, :, :] is the Forward state of the last layer
        # hidden[-1, :, :] is the Backward state of the last layer

        # Concatenate forward + backward
        hidden_cat = torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)
        cell_cat = torch.cat((cell[-2,:,:], cell[-1,:,:]), dim=1)

        # Squash them through the linear layer to match Decoder size
        # shape becomes [batch size, hid_dim]
        hidden = torch.tanh(self.fc_hidden(hidden_cat))
        cell = torch.tanh(self.fc_cell(cell_cat))

        # 4. Prepare for Decoder
        # The decoder expects shape [n_layers, batch, hid_dim].
        # Since we squashed it to [batch, hid_dim], we typically unsqueeze
        # to fit n_layers=1 for the decoder.
        # (Note: This simple implementation assumes Decoder n_layers=1)
        return hidden.unsqueeze(0), cell.unsqueeze(0)

class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, n_layers, dropout, pretrained_weights=None):
        super().__init__()
        self.output_dim = output_dim
        self.embedding = nn.Embedding(output_dim, emb_dim)

        if pretrained_weights is not None:
            self.embedding.weight.data.copy_(pretrained_weights)
            self.embedding.weight.requires_grad = False

        # Decoder remains UNIDIRECTIONAL
        self.lstm = nn.LSTM(emb_dim, hid_dim, n_layers, dropout=dropout, batch_first=True)

        self.fc_out = nn.Linear(hid_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input, hidden, cell):
        # input = [batch size]
        input = input.unsqueeze(1)
        embedded = self.dropout(self.embedding(input))

        # output = [batch size, 1, hid_dim]
        # hidden = [n_layers, batch, hid_dim]
        output, (hidden, cell) = self.lstm(embedded, (hidden, cell))

        prediction = self.fc_out(output.squeeze(1))
        return prediction, hidden, cell

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size = src.shape[0]
        trg_len = trg.shape[1]
        trg_vocab_size = self.decoder.output_dim

        outputs = torch.zeros(batch_size, trg_len, trg_vocab_size).to(self.device)

        # Encoder
        hidden, cell = self.encoder(src)

        input = trg[:, 0]

        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(input, hidden, cell)
            outputs[:, t, :] = output

            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1)
            input = trg[:, t] if teacher_force else top1

        return outputs

In [12]:
def predict_sentence(model, sentence, src_vocab, trg_vocab, device, max_len=50):
    model.eval()

    # Numericalize
    if isinstance(sentence, str):
        tokens = src_vocab.numericalize(sentence)
    else:
        return ""

    tokens = [src_vocab.stoi["<SOS>"]] + tokens + [src_vocab.stoi["<EOS>"]]
    src_tensor = torch.LongTensor(tokens).unsqueeze(0).to(device) # [1, seq_len]

    with torch.no_grad():
        hidden, cell = model.encoder(src_tensor)

    trg_indexes = [trg_vocab.stoi["<SOS>"]]

    # Decode step by step
    for _ in range(max_len):
        trg_tensor = torch.LongTensor([trg_indexes[-1]]).to(device)
        with torch.no_grad():
            output, hidden, cell = model.decoder(trg_tensor, hidden, cell)

        pred_token = output.argmax(1).item()
        trg_indexes.append(pred_token)

        if pred_token == trg_vocab.stoi["<EOS>"]:
            break

    # Convert back to words
    trg_tokens = [trg_vocab.itos[i] for i in trg_indexes]

    # Remove SOS and EOS for cleaner output
    return " ".join(trg_tokens[1:-1])

In [15]:
src_vocab = Vocabulary()
trg_vocab = Vocabulary()

print("Building vocabulary from Training data...")
src_vocab.build_vocabulary(train_df['text'].tolist())
trg_vocab.build_vocabulary(train_df['english_translation'].tolist())

print(f"Source Vocab Size: {len(src_vocab)}")
print(f"Target Vocab Size: {len(trg_vocab)}")

Building vocabulary from Training data...
Source Vocab Size: 8697
Target Vocab Size: 9394


In [16]:
BATCH_SIZE = 32

# 1. Train Loader (Shuffle = True)
train_dataset = ParallelDataset(train_df, 'text', 'english_translation', src_vocab, trg_vocab)
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=MyCollate(src_vocab.stoi["<PAD>"])
)

# 2. Validation Loader (Shuffle = False)
val_dataset = ParallelDataset(val_df, 'text', 'english_translation', src_vocab, trg_vocab)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=MyCollate(src_vocab.stoi["<PAD>"])
)

# 3. Test Loader (Shuffle = False)
test_dataset = ParallelDataset(test_df, 'text', 'english_translation', src_vocab, trg_vocab)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=MyCollate(src_vocab.stoi["<PAD>"])
)

In [17]:
from tqdm.auto import tqdm

def train_epoch(model, iterator, optimizer, criterion, clip):
    model.train()
    epoch_loss = 0

    pbar = tqdm(enumerate(iterator), total=len(iterator), desc="Training")

    for i, (src, trg) in pbar:
        src, trg = src.to(device), trg.to(device)
        optimizer.zero_grad()
        output = model(src, trg)

        output_dim = output.shape[-1]
        output = output[:, 1:].reshape(-1, output_dim)
        trg = trg[:, 1:].reshape(-1)

        loss = criterion(output, trg)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        epoch_loss += loss.item()

        pbar.set_postfix(loss=loss.item())

    return epoch_loss / len(iterator)

# --- 2. Update evaluate_loss ---
def evaluate_loss(model, iterator, criterion):
    model.eval()
    epoch_loss = 0

    pbar = tqdm(enumerate(iterator), total=len(iterator), desc="Validating")

    with torch.no_grad():
        for i, (src, trg) in pbar:
            src, trg = src.to(device), trg.to(device)

            output = model(src, trg, teacher_forcing_ratio=0)

            output_dim = output.shape[-1]
            output = output[:, 1:].reshape(-1, output_dim)
            trg = trg[:, 1:].reshape(-1)

            loss = criterion(output, trg)
            epoch_loss += loss.item()

            pbar.set_postfix(loss=loss.item())

    return epoch_loss / len(iterator)

ENC_EMB_DIM = 300
DEC_EMB_DIM = 300
HID_DIM = 256
N_LAYERS = 1
ENC_DROPOUT = 0.5
DEC_DROPOUT = 0.5

print("Loading Source (Indonesian) Embeddings...")
src_weights = load_pretrained_embeddings(src_vocab, 'wiki.id.vec', emb_dim=300)

print("Loading Target (English) Embeddings...")
trg_weights = load_pretrained_embeddings(trg_vocab, 'wiki.en.vec', emb_dim=300)

# 3. Initialize Model WITH Weights
enc = Encoder(len(src_vocab), ENC_EMB_DIM, HID_DIM, N_LAYERS, ENC_DROPOUT, pretrained_weights=src_weights)
dec = Decoder(len(trg_vocab), DEC_EMB_DIM, HID_DIM, N_LAYERS, DEC_DROPOUT, pretrained_weights=trg_weights)

model = Seq2Seq(enc, dec, device).to(device)

# 4. Optimizer & Loss
parameters_to_train = filter(lambda p: p.requires_grad, model.parameters())

optimizer = optim.Adam(parameters_to_train, lr=0.0005)

criterion = nn.CrossEntropyLoss(ignore_index=trg_vocab.stoi["<PAD>"])

# 5. Run Training
N_EPOCHS = 5
best_valid_loss = float('inf')

print(f"Starting training on {len(train_df)} examples, validating on {len(val_df)}...")

for epoch in range(N_EPOCHS):
    print(f"\nEpoch {epoch+1}/{N_EPOCHS}")

    train_loss = train_epoch(model, train_loader, optimizer, criterion, 1)
    valid_loss = evaluate_loss(model, val_loader, criterion)

    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), 'best-model.pt')
        print(f'\tResult: Train Loss: {train_loss:.3f} | Val. Loss: {valid_loss:.3f} (*) (Saved)')
    else:
        print(f'\tResult: Train Loss: {train_loss:.3f} | Val. Loss: {valid_loss:.3f}')

Loading Source (Indonesian) Embeddings...
Loading embeddings from wiki.id.vec...


Parsing Vectors: 0it [00:00, ?it/s]

Found 6258 / 8697 words in pretrained file.
Coverage: 71.96%
Loading Target (English) Embeddings...
Loading embeddings from wiki.en.vec...


Parsing Vectors: 0it [00:00, ?it/s]

Found 5524 / 9394 words in pretrained file.
Coverage: 58.80%


...

Validating:   0%|          | 0/32 [00:00<?, ?it/s]

	Result: Train Loss: 6.160 | Val. Loss: 6.402

Epoch 5/5


Training:   0%|          | 0/150 [00:00<?, ?it/s]

Validating:   0%|          | 0/32 [00:00<?, ?it/s]

	Result: Train Loss: 6.105 | Val. Loss: 6.428


In [19]:
model.load_state_dict(torch.load('best-model.pt'))

print("Generating predictions on Test Set...")
test_predictions = []

for src_text in test_df['text']:
    pred = predict_sentence(model, src_text, src_vocab, trg_vocab, device)
    test_predictions.append(pred)

test_df['LSTM_pred'] = test_predictions

evaluate_translation(test_df, 'english_translation', 'LSTM_pred', 'Bidirectional_LSTM')
evaluate_translation(test_df, 'better_english_translation', 'LSTM_pred', 'Bidirectional_LSTM')

Generating predictions on Test Set...

Calculating Evaluation Metrics...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


------------------------------
Model: Bidirectional_LSTM
BLEU Score: 0.22
Meteor Score: 0.06
------------------------------

Calculating Evaluation Metrics...


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


------------------------------
Model: Bidirectional_LSTM
BLEU Score: 0.23
Meteor Score: 0.06
------------------------------


In [ ]:
test_df.to_csv("LSTM_results2.csv")

## Helsinki-NLP

In [20]:
test_df = pd.read_csv('test_merged2.csv')

### Baseline

In [ ]:
MODEL_NAME = "Helsinki-NLP/opus-mt-id-en"
BATCH_SIZE = 32
MAX_LENGTH = 128
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)

def translate_batch(texts, model, tokenizer, device):
    inputs = tokenizer(
        texts, return_tensors="pt", padding=True, truncation=True, max_length=MAX_LENGTH
    ).to(device)

    with torch.no_grad():
        translated = model.generate(**inputs, max_new_tokens=MAX_LENGTH, num_beams=4, early_stopping=True)

    return tokenizer.batch_decode(translated, skip_special_tokens=True)

source_texts = test_df['text'].tolist()

dataset = TranslationDataset(source_texts)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

all_predictions = []
for batch in tqdm(dataloader, desc="Translating Test Set"):
    if not isinstance(batch, list): batch = list(batch)
    preds = translate_batch(batch, model, tokenizer, device)
    all_predictions.extend(preds)

test_df['pred_helsinki_baseline'] = all_predictions

evaluate_translation(
    df=test_df,
    target_col='english_translation',
    output_col='pred_helsinki_baseline',
    model_name=f"{MODEL_NAME} (Baseline)"
)

evaluate_translation(
    df=test_df,
    target_col='better_english_translation',
    output_col='pred_helsinki_baseline',
    model_name=f"{MODEL_NAME} (Gold Standard)"
)

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

source.spm:   0%|          | 0.00/801k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/796k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...


Calculating Evaluation Metrics...


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


------------------------------
Model: Helsinki-NLP/opus-mt-id-en (Baseline)
BLEU Score: 10.53
Meteor Score: 0.36
------------------------------

Calculating Evaluation Metrics...


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


------------------------------
Model: Helsinki-NLP/opus-mt-id-en (Gold Standard)
BLEU Score: 9.14
Meteor Score: 0.33
------------------------------


### Fine Tuning

In [21]:
MODEL_CHECKPOINT = "Helsinki-NLP/opus-mt-id-en"
OUTPUT_DIR = "./fine_tuned_helsinki_tourism"
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
NUM_EPOCHS = 3
WEIGHT_DECAY = 0.01
MAX_INPUT_LENGTH = 128
MAX_TARGET_LENGTH = 128

data_files_train = {
    "train": "train2.csv",
    "validation": "val2.csv"
}

dataset = load_dataset("csv", data_files=data_files_train)


tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def preprocess_function(examples):
    inputs = [str(ex) for ex in examples['text']]
    targets = [str(ex) for ex in examples['english_translation']]

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LENGTH,
        truncation=True
    )

    labels = tokenizer(
        text_target=targets,
        max_length=MAX_TARGET_LENGTH,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = dataset.map(preprocess_function, batched=True)

metric = evaluate.load("sacrebleu")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [[label.strip()] for label in decoded_labels]

    result = metric.compute(predictions=decoded_preds, references=decoded_labels)
    return {"bleu": result["score"]}

print(f"Loading model: {MODEL_CHECKPOINT}...")
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_CHECKPOINT)

args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="bleu",
    greater_is_better=True,
    save_total_limit=1,
    save_steps=50000,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    weight_decay=WEIGHT_DECAY,
    num_train_epochs=NUM_EPOCHS,
    predict_with_generate=True,
    fp16=True,
    push_to_hub=False,
    report_to="none"
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

print("Starting Fine-Tuning...")
trainer.train()

print(f"Saving fine-tuned model to {OUTPUT_DIR}...")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

source.spm:   0%|          | 0.00/801k [00:00<?, ?B/s]

...

Starting Fine-Tuning...


Epoch,Training Loss,Validation Loss,Bleu
1,No log,1.348825,33.401481
2,1.590600,1.270848,35.131071
3,1.590600,1.252382,35.639419


/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 512, 'num_beams': 6, 'bad_words_ids': [[54795]]}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(
There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.encoder.embed_positions.weight', 'model.decoder.embed_tokens.weight', 'model.decoder.embed_positions.weight', 'lm_head.weight'].


Saving fine-tuned model to ./fine_tuned_helsinki_tourism...


('./fine_tuned_helsinki_tourism/tokenizer_config.json',
 './fine_tuned_helsinki_tourism/special_tokens_map.json',
 './fine_tuned_helsinki_tourism/vocab.json',
 './fine_tuned_helsinki_tourism/source.spm',
 './fine_tuned_helsinki_tourism/target.spm',
 './fine_tuned_helsinki_tourism/added_tokens.json')

In [ ]:
OUTPUT_DIR = "./fine_tuned_helsinki_tourism"
print("Loading Fine-Tuned Model for Evaluation...")
finetuned_model = AutoModelForSeq2SeqLM.from_pretrained(OUTPUT_DIR).to("cuda")
finetuned_tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)

def predict_finetuned(texts):
    inputs = finetuned_tokenizer(
        texts, return_tensors="pt", padding=True, truncation=True, max_length=128
    ).to("cuda")

    with torch.no_grad():
        translated = finetuned_model.generate(**inputs, max_new_tokens=128, num_beams=4, early_stopping=True)

    return finetuned_tokenizer.batch_decode(translated, skip_special_tokens=True)

print("Predicting on Test Set...")
test_texts = test_df['text'].tolist()
dataset = TranslationDataset(test_texts)
dataloader = DataLoader(dataset, batch_size=32, shuffle=False)

all_preds = []
for batch in tqdm(dataloader):
    if not isinstance(batch, list): batch = list(batch)
    all_preds.extend(predict_finetuned(batch))

test_df['pred_helsinki_finetuned'] = all_preds

evaluate_translation(
    df=test_df,
    target_col='english_translation',
    output_col='pred_helsinki_finetuned',
    model_name="Helsinki Fine-Tuned"
)

evaluate_translation(
    df=test_df,
    target_col='better_english_translation',
    output_col='pred_helsinki_finetuned',
    model_name="Helsinki Fine-Tuned (Gold Standard)"
)

Loading Fine-Tuned Model for Evaluation...
Predicting on Test Set...


  0%|          | 0/7 [00:00<?, ?it/s]


Calculating Evaluation Metrics...


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


------------------------------
Model: Helsinki Fine-Tuned
BLEU Score: 33.22
Meteor Score: 0.66
------------------------------

Calculating Evaluation Metrics...


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


------------------------------
Model: Helsinki Fine-Tuned (Gold Standard)
BLEU Score: 28.76
Meteor Score: 0.61
------------------------------


In [22]:
import shutil
import os
from google.colab import files

# Configuration
FOLDER_TO_ZIP = "./fine_tuned_helsinki_tourism"  # The folder you want to download
OUTPUT_FILENAME = "fine_tuned_helsinki_tourism"  # The name of the zip file

print(f"Zipping folder '{FOLDER_TO_ZIP}'...")

# Create the zip archive
shutil.make_archive(OUTPUT_FILENAME, 'zip', FOLDER_TO_ZIP)

print(f"Created {OUTPUT_FILENAME}.zip")
print("Starting download...")

# Trigger download
files.download(f"{OUTPUT_FILENAME}.zip")

Zipping folder './fine_tuned_helsinki_tourism'...
Created fine_tuned_helsinki_tourism.zip
Starting download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
test_df.to_csv('helsinki_results2.csv', index=False)

## NLBB

In [23]:
test_df = pd.read_csv('test_merged2.csv')

### Baseline

In [ ]:
MODEL_NAME = "facebook/nllb-200-distilled-600M"
SRC_LANG = "ind_Latn"
TGT_LANG = "eng_Latn"

BATCH_SIZE = 16
MAX_LENGTH = 128
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Using device: {device}")
print(f"Loading model: {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)

def translate_batch_nllb(texts, model, tokenizer, device):
    tokenizer.src_lang = SRC_LANG

    inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH
    ).to(device)

    forced_bos_token_id = tokenizer.convert_tokens_to_ids(TGT_LANG)

    with torch.no_grad():
        translated_tokens = model.generate(
            **inputs,
            forced_bos_token_id=forced_bos_token_id,
            max_new_tokens=MAX_LENGTH,
            num_beams=4,
            early_stopping=True
        )

    return tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)

print(f"Running NLLB baseline on {len(test_df)} rows...")

dataset = TranslationDataset(test_df['text'].tolist())
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

all_predictions = []

for batch_texts in tqdm(dataloader, desc="Translating"):
    if not isinstance(batch_texts, list):
        batch_texts = list(batch_texts)

    batch_preds = translate_batch_nllb(batch_texts, model, tokenizer, device)
    all_predictions.extend(batch_preds)

test_df['pred_nllb_baseline'] = all_predictions

evaluate_translation(
    df=test_df,
    target_col='english_translation',
    output_col='pred_nllb_baseline',
    model_name="NLLB-200 Baseline"
)

evaluate_translation(
    df=test_df,
    target_col='better_english_translation',
    output_col='pred_nllb_baseline',
    model_name="NLLB-200 Baseline (Gold Standard)"
)

Using device: cuda
Loading model: facebook/nllb-200-distilled-600M...


tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...


Calculating Evaluation Metrics...


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


------------------------------
Model: NLLB-200 Baseline
BLEU Score: 14.10
Meteor Score: 0.44
------------------------------

Calculating Evaluation Metrics...


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


------------------------------
Model: NLLB-200 Baseline (Gold Standard)
BLEU Score: 12.11
Meteor Score: 0.40
------------------------------


### Fine tuning

In [28]:
MODEL_NAME = "facebook/nllb-200-distilled-600M"
OUTPUT_DIR = "./fine_tuned_nllb_tourism"
SRC_LANG = "ind_Latn"
TGT_LANG = "eng_Latn"
BATCH_SIZE = 8
LEARNING_RATE = 2e-5
NUM_EPOCHS = 3
WEIGHT_DECAY = 0.01
MAX_LENGTH = 128

data_files = {
    "train": "train2.csv",
    "validation": "val2.csv",
}
dataset = load_dataset("csv", data_files=data_files)

print(f"Loading tokenizer: {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
if tokenizer.bos_token is None:
    tokenizer.bos_token = tokenizer.eos_token
tokenizer.src_lang = "ind_Latn"
tokenizer.tgt_lang = "eng_Latn"

def preprocess_nllb(examples):
    inputs = examples['text']
    targets = examples['english_translation']

    tokenizer.src_lang = SRC_LANG

    model_inputs = tokenizer(
        inputs,
        text_target=targets,
        max_length=MAX_LENGTH,
        truncation=True
    )

    return model_inputs

print("Tokenizing data...")
tokenized_datasets = dataset.map(
    preprocess_nllb,
    batched=True
)

metric = evaluate.load("sacrebleu")

def compute_metrics_nllb(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple): preds = preds[0]

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [[label.strip()] for label in decoded_labels]

    result = metric.compute(predictions=decoded_preds, references=decoded_labels)
    return {"bleu": result["score"]}

print(f"Loading model: {MODEL_NAME}...")
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

# Force Target Language
forced_bos_id = tokenizer.convert_tokens_to_ids(TGT_LANG)
model.config.forced_bos_token_id = forced_bos_id

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="bleu",
    greater_is_better=True,
    save_total_limit=1,
    save_steps=50000,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    weight_decay=WEIGHT_DECAY,
    num_train_epochs=NUM_EPOCHS,
    predict_with_generate=True,
    fp16=True if torch.cuda.is_available() else False,
    push_to_hub=False,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics_nllb
)

print("Starting Fine-Tuning NLLB...")
trainer.train()

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Model saved to {OUTPUT_DIR}")

Loading tokenizer: facebook/nllb-200-distilled-600M...
Tokenizing data...


Map:   0%|          | 0/4800 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Loading model: facebook/nllb-200-distilled-600M...
Starting Fine-Tuning NLLB...


Epoch,Training Loss,Validation Loss,Bleu
1,1.082200,0.856401,39.574942
2,0.867600,0.815200,41.320973
3,0.794000,0.806052,41.336568


/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 200, 'forced_bos_token_id': 256047}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(
There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


Model saved to ./fine_tuned_nllb_tourism


In [ ]:
OUTPUT_DIR = "./fine_tuned_nllb_tourism"
TGT_LANG = "eng_Latn" # Target Language for NLLB (English)

print(f"Loading Fine-Tuned NLLB Model from {OUTPUT_DIR}...")
finetuned_model = AutoModelForSeq2SeqLM.from_pretrained(OUTPUT_DIR).to("cuda")
finetuned_tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)

forced_bos_id = finetuned_tokenizer.convert_tokens_to_ids(TGT_LANG)

def predict_finetuned_nllb(texts):
    inputs = finetuned_tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=128
    ).to("cuda")

    with torch.no_grad():
        translated = finetuned_model.generate(
            **inputs,
            forced_bos_token_id=forced_bos_id,
            max_new_tokens=128,
            num_beams=4,
            early_stopping=True
        )

    return finetuned_tokenizer.batch_decode(translated, skip_special_tokens=True)

print("Predicting on Test Set (NLLB)...")
test_texts = test_df['text'].tolist()
dataset = TranslationDataset(test_texts)
dataloader = DataLoader(dataset, batch_size=16, shuffle=False)

all_preds = []
for batch in tqdm(dataloader):
    if not isinstance(batch, list): batch = list(batch)
    all_preds.extend(predict_finetuned_nllb(batch))

# Save to specific column
test_df['pred_nllb_finetuned'] = all_preds

# Evaluate
evaluate_translation(
    df=test_df,
    target_col='english_translation',
    output_col='pred_nllb_finetuned',
    model_name="NLLB Fine-Tuned"
)

evaluate_translation(
    df=test_df,
    target_col='better_english_translation',
    output_col='pred_nllb_finetuned',
    model_name="NLLB Fine-Tuned (Gold Standard)"
)

Loading Fine-Tuned NLLB Model from ./fine_tuned_nllb_tourism...
Predicting on Test Set (NLLB)...


  0%|          | 0/13 [00:00<?, ?it/s]


Calculating Evaluation Metrics...


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


------------------------------
Model: NLLB Fine-Tuned
BLEU Score: 39.04
Meteor Score: 0.71
------------------------------

Calculating Evaluation Metrics...


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


------------------------------
Model: NLLB Fine-Tuned (Gold Standard)
BLEU Score: 33.50
Meteor Score: 0.64
------------------------------


In [ ]:
test_df.to_csv('nllb_results2.csv', index=False)

In [29]:
import os
import zipfile

# Folder model kamu
folder = "/content/fine_tuned_nllb_tourism"

# File-file essential
essential_files = [
    "model.safetensors",
    "config.json",
    "generation_config.json",
    "sentencepiece.bpe.model",
    "special_tokens_map.json",
    "tokenizer.json",
    "tokenizer_config.json"
]

# Nama output ZIP
zip_path = "/content/fine_tuned_nllb_tourism_essential2.zip"

# Buat ZIP
with zipfile.ZipFile(zip_path, 'w') as z:
    for fname in essential_files:
        full_path = os.path.join(folder, fname)
        if os.path.exists(full_path):
            z.write(full_path, arcname=fname)
        else:
            print(f"File tidak ditemukan: {fname}")

print("ZIP selesai dibuat:", zip_path)


ZIP selesai dibuat: /content/fine_tuned_nllb_tourism_essential2.zip


In [ ]:
# !mv fine_tuned_nllb_tourism_essential.zip /content/drive/MyDrive/

## mBART

In [30]:
test_df = pd.read_csv('test_merged2.csv')

In [31]:
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast
from tqdm.auto import tqdm

# 1. Setup Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# 2. Load Model & Tokenizer
model_name = "facebook/mbart-large-50-many-to-many-mmt"
print("Loading mBART model... (This might take a moment)")
tokenizer = MBart50TokenizerFast.from_pretrained(model_name)
model = MBartForConditionalGeneration.from_pretrained(model_name).to(device)

Using device: cuda
Loading mBART model... (This might take a moment)


tokenizer_config.json:   0%|          | 0.00/529 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/649 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/261 [00:00<?, ?B/s]

In [ ]:
def generate_predictions(model, tokenizer, df, src_col, batch_size=8):
    model.eval()
    predictions = []

    tokenizer.src_lang = "id_ID"
    lang_code_to_id = tokenizer.lang_code_to_id["en_XX"]

    # Create batches
    src_texts = df[src_col].tolist()
    total_batches = (len(src_texts) + batch_size - 1) // batch_size

    print(f"Generating translations for {len(src_texts)} rows...")

    for i in tqdm(range(0, len(src_texts), batch_size), total=total_batches):
        batch = src_texts[i : i + batch_size]

        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)

        with torch.no_grad():
            generated_tokens = model.generate(
                **inputs,
                forced_bos_token_id=lang_code_to_id, # Force output to be English
                max_length=128,
                num_beams=5,
                early_stopping=True
            )

        decoded = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
        predictions.extend(decoded)

    return predictions

baseline_preds = generate_predictions(model, tokenizer, test_df, 'text', batch_size=8)
test_df['mbart_baseline'] = baseline_preds

evaluate_translation(test_df, 'english_translation', 'mbart_baseline', 'mBART_Baseline')
evaluate_translation(test_df, 'better_english_translation', 'mbart_baseline', 'mBART_Baseline')

Generating translations for 200 rows...


  0%|          | 0/25 [00:00<?, ?it/s]


Calculating Evaluation Metrics...


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


------------------------------
Model: mBART_Baseline
BLEU Score: 12.45
Meteor Score: 0.40
------------------------------

Calculating Evaluation Metrics...


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


------------------------------
Model: mBART_Baseline
BLEU Score: 11.20
Meteor Score: 0.39
------------------------------


In [33]:
MODEL_CHECKPOINT = "facebook/mbart-large-50-many-to-many-mmt"
OUTPUT_DIR = "./mbart_fine_tuned"

SRC_LANG = "id_ID"
TGT_LANG = "en_XX"

BATCH_SIZE = 4
LEARNING_RATE = 2e-5
NUM_EPOCHS = 3
WEIGHT_DECAY = 0.01
MAX_LENGTH = 128
GRADIENT_ACCUMULATION = 4

data_files = {
    "train": "train2.csv",
    "validation": "val2.csv",
}
dataset = load_dataset("csv", data_files=data_files)

print(f"Loading tokenizer: {MODEL_CHECKPOINT}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT, use_fast=False)

tokenizer.src_lang = SRC_LANG
tokenizer.tgt_lang = TGT_LANG

def preprocess_function(examples):
    inputs = [str(ex) for ex in examples['text']]
    targets = [str(ex) for ex in examples['english_translation']]

    # Tokenize Inputs
    model_inputs = tokenizer(
        inputs,
        max_length=MAX_LENGTH,
        truncation=True
    )

    # Tokenize Targets (Labels)
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            targets,
            max_length=MAX_LENGTH,
            truncation=True
        )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("Tokenizing data...")
tokenized_datasets = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset["train"].column_names
)

metric = evaluate.load("sacrebleu")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple): preds = preds[0]

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [pred.strip() for pred in decoded_preds]
    # mBART adds the language code token (e.g., 'en_XX') to the reference,
    # so we must add it back manually for accurate BLEU calculation.
    decoded_labels = [[TGT_LANG + " " + label.strip()] for label in decoded_labels]

    result = metric.compute(predictions=decoded_preds, references=decoded_labels)
    return {"bleu": result["score"]}

print(f"Loading model: {MODEL_CHECKPOINT}...")
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_CHECKPOINT)

model.config.forced_bos_token_id = tokenizer.lang_code_to_id[TGT_LANG]

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="bleu",
    greater_is_better=True,
    save_total_limit=1,
    save_steps=50000,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    weight_decay=WEIGHT_DECAY,
    num_train_epochs=NUM_EPOCHS,
    predict_with_generate=True,
    fp16=True if torch.cuda.is_available() else False,
    push_to_hub=False,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

# --- 7. START TRAINING ---
print("Starting mBART Fine-Tuning...")
trainer.train()

# --- 8. SAVE ---
print(f"Saving fine-tuned model to {OUTPUT_DIR}...")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

Loading tokenizer: facebook/mbart-large-50-many-to-many-mmt...
Tokenizing data...


Map:   0%|          | 0/4800 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Loading model: facebook/mbart-large-50-many-to-many-mmt...


/tmp/ipython-input-2287385612.py:102: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


...

/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 200, 'early_stopping': True, 'num_beams': 5, 'forced_bos_token_id': 250004}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Bleu
1,No log,0.834219,38.278436
2,0.870100,0.795737,39.584873
3,0.870100,0.809163,39.532221


There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


Saving fine-tuned model to ./mbart_fine_tuned...


('./mbart_fine_tuned/tokenizer_config.json',
 './mbart_fine_tuned/special_tokens_map.json',
 './mbart_fine_tuned/sentencepiece.bpe.model',
 './mbart_fine_tuned/added_tokens.json')

In [11]:
OUTPUT_DIR = "./mbart_fine_tuned"
SRC_LANG = "id_ID"
TGT_LANG = "en_XX"
MAX_LENGTH = 128
BATCH_SIZE = 16

device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Using device: {device}")
print(f"Loading Fine-Tuned mBART model from: {OUTPUT_DIR}...")

finetuned_tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)
finetuned_model = AutoModelForSeq2SeqLM.from_pretrained(OUTPUT_DIR).to(device)

forced_bos_id = finetuned_tokenizer.lang_code_to_id[TGT_LANG]

def predict_finetuned_mbart(texts):
    # CRITICAL: Set source language for tokenization
    finetuned_tokenizer.src_lang = SRC_LANG

    inputs = finetuned_tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH
    ).to(device)

    with torch.no_grad():
        translated = finetuned_model.generate(
            **inputs,
            forced_bos_token_id=forced_bos_id, # Force English Output
            max_new_tokens=MAX_LENGTH,
            num_beams=4,
            early_stopping=True
        )

    return finetuned_tokenizer.batch_decode(translated, skip_special_tokens=True)

print("Generating translations with Fine-Tuned mBART...")

test_texts = test_df['text'].tolist()
dataset = TranslationDataset(test_texts)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

all_predictions = []
for batch in tqdm(dataloader, desc="Translating"):
    if not isinstance(batch, list): batch = list(batch)
    all_predictions.extend(predict_finetuned_mbart(batch))

# 5. SAVE & EVALUATE
test_df['mbart_finetuned'] = all_predictions

# Evaluate using your existing function
evaluate_translation(
    df=test_df,
    target_col='english_translation',
    output_col='mbart_finetuned',
    model_name="mBART Fine-Tuned"
)

evaluate_translation(
    df=test_df,
    target_col='better_english_translation',
    output_col='mbart_finetuned',
    model_name="mBART Fine-Tuned"
)


Using device: cuda
Loading Fine-Tuned mBART model from: ./mbart_fine_tuned...
🔄 Generating translations with Fine-Tuned mBART...


Translating:   0%|          | 0/13 [00:00<?, ?it/s]


Calculating Evaluation Metrics...


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


------------------------------
Model: mBART Fine-Tuned
BLEU Score: 39.25
Meteor Score: 0.69
------------------------------

Calculating Evaluation Metrics...


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


------------------------------
Model: mBART Fine-Tuned
BLEU Score: 34.75
Meteor Score: 0.65
------------------------------


In [12]:
test_df.to_csv("mbart_results2.csv")
!mv mbart_results2.csv /content/drive/MyDrive/

In [36]:
import os
import zipfile
import shutil

folder = "./mbart_fine_tuned"

zip_path = "./mbart_fine_tuned_essential2.zip"

essential_files = [
    "model.safetensors",
    "config.json",
    "generation_config.json",
    "sentencepiece.bpe.model",
    "special_tokens_map.json",
    "tokenizer.json",
    "tokenizer_config.json"
]

try:
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
        for fname in essential_files:
            full_path = os.path.join(folder, fname)
            if os.path.exists(full_path):
                # Tulis file ke ZIP, dengan nama file asli (arcname)
                z.write(full_path, arcname=fname)

    print("ZIP selesai dibuat:", zip_path)

except Exception as e:
    print(f"ERROR saat membuat ZIP: {e}")


ZIP selesai dibuat: ./mbart_fine_tuned_essential2.zip


In [35]:
# !mv mbart_fine_tuned_essential.zip /content/drive/MyDrive/
files.download(f"mbart_fine_tuned_essential2.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>